# Lesson 19 — Stochastic Programming

## Learning objectives

1. Distinguish 2-stage from multi-stage SP.
2. Use sample average approximation (SAA) for SP.
3. Recognize chance constraints and CVaR.
4. Connect to robust optimization (Lesson 25).

## 1. The two-stage SP

$$\min_x c^\top x + \mathbb{E}_\xi[Q(x, \xi)] \quad \text{s.t.} \quad x \in \mathcal{X}$$

where $Q(x, \xi) = \min_y \{q^\top y : Wy = h - Tx, y \ge 0\}$ is the second-stage value under realization $\xi = (q, h, T, W)$.

If $\xi$ has finitely many scenarios $s = 1, \dots, S$ with probabilities $p_s$, the **extensive form** is one big LP. With many scenarios, decomposition (Benders, Lesson 18) is essential. {cite:p}`Birge2011` is the standard reference.

## 2. Sample average approximation (SAA)

For continuous $\xi$, sample $N$ realizations and solve the *deterministic* SP with these $N$ scenarios. As $N \to \infty$, the optimal value converges (SAA consistency). {cite:p}`Shapiro2009` Ch 5.

In practice: sample 100, 500, 1000 scenarios; solve; compare optimal values to gauge stability.

## 3. Chance constraints

$$\Pr_\xi(\xi^\top x \le b) \ge 1 - \epsilon.$$

For Gaussian $\xi$, equivalent to a deterministic constraint:

$$\bar \xi^\top x + \Phi^{-1}(1 - \epsilon) \sqrt{x^\top \Sigma x} \le b,$$

an SOCP! For non-Gaussian, harder; SAA + integer indicators is one way. {cite:p}`Birge2011`.

## 4. CVaR

$$\text{CVaR}_\alpha(L) = \mathbb{E}[L \mid L \ge \text{VaR}_\alpha(L)].$$

For a random loss $L(x, \xi)$, minimizing CVaR is convex in $x$ when $L$ is convex. Replaces the non-convex VaR; widely used in risk-aware optimization.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# Newsvendor: order x today, demand D ~ uniform(50, 150)
# Cost: 2*x today, revenue 5*min(x, D), salvage 1*max(0, x - D)
np.random.seed(0)
N = 200
D = np.random.uniform(50, 150, size=N)

m = do.Model("newsvendor")
x = m.continuous("x", lb=0, ub=200)
y = m.continuous("y", shape=(N,), lb=0)             # sold
z = m.continuous("z", shape=(N,), lb=0)             # salvage
for s in range(N):
    m.subject_to(y[s] <= x)
    m.subject_to(y[s] <= D[s])
    m.subject_to(z[s] == x - y[s])      # equality! z = unsold = salvage quantity
# Salvage z is recovered revenue (1 per unit), NOT a cost — both 5*y and 1*z carry the same sign.
m.minimize(2 * x - sum(5 * y[s] + 1 * z[s] for s in range(N)) / N)
r = m.solve()
print(f"order quantity: {r.value(x):.2f}, expected profit: {-r.objective:.2f}")


## 5. Multi-stage SP

For $T$ stages, decisions are recourse functions $x_t(\xi_{1:t})$. Number of scenarios grows exponentially in $T$. Approximation strategies:

- **Scenario tree** (small finite tree).
- **SDDP** (stochastic dual dynamic programming) — Benders cuts in time {cite:p}`Birge2011`.

Multi-stage SP is the bread and butter of energy and finance applications.

## References
{cite:p}`Birge2011,Shapiro2009,Wolsey1998`.